# MetaMARL tutorial: bilevel mechanism design in a shared fishery

This notebook introduces the main MetaMARL abstractions through the **fishery benchmark**.

We will build a two-level learning problem:

- a **society** of fishers learns how much to harvest and how much restoration effort to apply;
- a **regulator** learns a quota mechanism that intervenes on the fishers' harvest decisions;
- the two levels are coupled through the fish stock: the society changes the stock directly, while the regulator acts on it only through the quota mechanism that is applied inside the society's environment.

The goal of the notebook is not to hide the configuration behind helper scripts. Instead, it shows how the pieces fit together so that you can replace the optimizer, mechanism, environment, reporter, or agent population without changing the overall workflow.

> **Scale note.** The notebook runs at a deliberately small scale, so that it executes end to end in about one minute. Section 10 lists the full-scale settings of the original experiment, so that you can launch the real run by changing five named constants.

## 1. The benchmark in one picture

At the society level, each fisher observes the shared resource and produces policy actions. The benchmark maps those actions through fishery mechanisms such as `Fishing` and `Restore`.

At the regulator level, a second optimizer controls an institutional mechanism such as `Quota`. The quota targets the fishers' harvest mechanism rather than replacing the fishers' policy.

Conceptually:

$$
\text{regulator parameters}
\;\longrightarrow\;
\text{institutional mechanism}
\;\longrightarrow\;
\text{society learning}
\;\longrightarrow\;
\text{fishery trajectory}
\;\longrightarrow\;
\text{regulator objective}.
$$

This is why MetaMARL exposes the two levels separately as:

```text
BilevelConfig
├── regulator(...)
└── society(...)
```

## 2. Fishery dynamics

Let $B_t$ denote the fish-stock biomass, $K$ the carrying capacity, $r$ the intrinsic growth rate, and $p$ the Pella–Tomlinson shape parameter. The surplus production of the stock is

$$
g(B_t)
=
\frac{r}{p}B_t
\left[
1-\left(\frac{B_t}{K}\right)^p
\right],
$$

and the benchmark advances the stock with the discrete-time update

$$
B_{t+1}
=
\max\!\left(
B_t
+
g(B_t)
+
R_t
+
\varepsilon_t
-
C_t,\;
0
\right),
\qquad
\varepsilon_t = \sigma\,\xi_t\,B_t,
\quad
\xi_t \sim \mathcal N(0,1),
$$

where:

- $\varepsilon_t$ is Gaussian process noise proportional to the stock, with intensity $\sigma$ (the `sigma` entry of the ecology);
- $R_t$ is the total restoration added by the fishers;
- $C_t$ is the total catch actually delivered to the fishers, which can never exceed $B_t$.

When $p=1$, the biological growth term reduces to the Schaefer logistic form.

The fishers act simultaneously on the stock $B_t$ that the step starts with. Each fisher $i$ requests a catch $Q_{i,t}$, and the transition adds the requests up to $Q_t=\sum_i Q_{i,t}$. When the stock cannot cover the total request, every fisher receives the same fraction of its request, a *pro-rata* share:

$$
C_{i,t}
=
Q_{i,t}\,\min\!\left(1,\frac{B_t}{Q_t}\right),
\qquad
C_t=\sum_i C_{i,t}=\min\left(Q_t,\,B_t\right).
$$

The environment also computes standard reference points:

$$
B_{\mathrm{MSY}}
=
K\left(\frac{1}{p+1}\right)^{1/p},
$$

$$
\mathrm{MSY}
=
\frac{rK}{(p+1)^{(p+1)/p}},
$$

and

$$
F_{\mathrm{MSY}}
=
\frac{\mathrm{MSY}}{B_{\mathrm{MSY}}}.
$$

The surplus-production model follows Schaefer (1954) and, for the shape parameter $p$, Pella and Tomlinson (1969); the references are listed at the end of the notebook. The module docstring of `examples/bilevel_fishery/regulated_env.py` is the source of truth for these equations.

### 2.1 Fisher actions and rewards

The policy does not directly emit biomass. Its raw action is decoded by a mechanism.

For harvesting, the benchmark squashes an unbounded scalar $z_{i,t}$ through

$$
h_{i,t}
=
\operatorname{sigmoid}\left(\frac{z_{i,t}}{4}\right),
$$

the fraction of the fisher's maximal catch that it wants to take. The regulator's quota acts before the fishers: it lowers the raw action $z_{i,t}$, so the fraction $h_{i,t}$ that is decoded is the one that remains after the quota. The request is then that fraction of the fisher's catch capacity,

$$
Q_{i,t}
=
h_{i,t}
\left(
m F_{\mathrm{MSY}}
\frac{B_t}{N}
\right),
$$

where $m$ is `unregulated_f_multiplier` and $N$ is the number of fishers.

Choosing $m>1$ is important for the benchmark: it allows the unregulated society to request harvest above the sustainable reference level, so the institutional mechanism has a non-trivial problem to solve.

The reward of fisher $i$ at step $t$ is the harvest fraction $h_{i,t}$ itself, taken after the quota has lowered it and before the pro-rata rationing of Section 2. A fisher is therefore paid for what it was allowed to request, even on a step where the stock could not cover every request.

Restoration is decoded similarly:

$$
R_{i,t}
=
\rho K
\frac{e_{i,t}}{N},
\qquad
e_{i,t}=\operatorname{sigmoid}\left(\frac{z^{\mathrm{restore}}_{i,t}}{4}\right),
$$

where $\rho$ is `restoration_effectiveness` and $R_t=\sum_i R_{i,t}$. The environment itself charges no cost for restoration.

## 3. The package mental model

There are four objects to keep separate.

**Environment.** `FisheryRegulatedEnv` owns the ecological state and transition dynamics.

**Agent configuration.** `AgentConfig` (and benchmark-specific subclasses such as `FishermanConfig`) describes who acts, which policy they use, how many copies exist, and which mechanisms belong to that agent.

**Mechanism.** A mechanism decodes or transforms a decision before it affects the system. In this benchmark:

- `Fishing` converts a policy action into harvest pressure;
- `Restore` converts a policy action into biomass restoration;
- `Quota` is owned by the regulator and targets the fishers' `"harvest"` mechanism.

**Optimizer configuration.** The society and regulator can use different optimizers. Here the regulator uses Evolution Strategies (ES; Salimans et al., 2017), while the society uses APPO, RLlib's asynchronous proximal policy optimization, which corrects off-policy samples with V-trace (Espeholt et al., 2018).

The bilevel container composes two optimizer configurations around a shared world, and it asks little of them. The outer optimizer must return a summary from `train()` with the keys `episodes`, `converged`, `best_mechanism` and `best_fitness`, and the inner optimizer must report a `batch_capacity`, the number of mechanism candidates it evaluates in parallel, which sizes the population of the outer optimizer. ES is the only outer optimizer in the package today.

## 4. Imports

The tutorial uses the benchmark objects already provided under `examples.bilevel_fishery`.

The `examples/` directory is not an installed package, so the first code cell puts the repository root on the import path.

In [ ]:
import sys
from pathlib import Path

# `examples/` is not an installed package: put the repository root on the path.
REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import logging

import numpy as np
import ray
from gymnasium import spaces

from core.agents.base import AgentConfig
from core.callbacks import log_and_report_episode_metrics, tag_episode_with_env_idx
from core.mechanism.algorithms.quota import Quota
from core.optimizers.appo.config import APPOptimizerConfig
from core.optimizers.bilevel import BilevelConfig
from core.optimizers.es.config import ESConfig
from core.reporting.wandb import WandbConfig
from examples.bilevel_fishery.metric_schema import FisheryMetricSchema
from examples.bilevel_fishery.queries import ES_QUERIES, INNER_QUERIES
from examples.bilevel_fishery.regulated_env import (
    FishermanConfig,
    FisheryRegulatedEnv,
    FishingConfig,
    RestoreConfig,
)
from examples.bilevel_fishery.regulator_env import FisheryRegulatorEnv

## 5. Shared experiment constants

The first five constants set the scale of the run. They are reduced here so that the whole notebook executes in about a minute; Section 10 gives the full-scale values of the original experiment.

The ecology of the fishery is kept in one dictionary that only the society environment reads. The regulator environment reads no ecological constant. It reads two settings of its objective, and two optional settings that this notebook leaves at their defaults: `aggregation_status` (`"eval"`, the split of the inner metrics that is scored) and `fitness_tail_steps` (50, the number of final steps of each episode on which the fitness is computed; with `HORIZON = 50` that is the whole episode).

The fitness of a candidate quota is

$$
\text{fitness}
=
\text{harvest score}
+
w\,\overline{B/K},
$$

where the harvest score is the realized harvest over the final steps divided by the maximum sustainable yield, $\overline{B/K}$ is the mean normalized biomass over the same steps, and $w$ is `sustainability_weight`. The other setting, `sustainability_threshold`, is the normalized biomass below which a step counts as collapsed; it enters the collapse rate that the regulator logs, and not the fitness. In particular, the carrying capacity $K$ is a society setting.

The weight of 2, the threshold of 0.20, the ecology and the settings of the evolution strategy are heuristic choices of this example. None of them is calibrated to data or derived from a theory, and the objective itself is a modelling choice: a weighted log-utility was considered in the code and not kept.

In [ ]:
# Scale of the run: small by default, full-scale values in Section 10.
ES_GENERATIONS = 12
APPO_ITERATIONS = 8
HORIZON = 50
NUM_FISHERS = 4
EVALUATION_SEEDS = 1

K = 5_000
SEED = 42

ECOLOGY = {
    "r": 0.3,
    "K": K,
    "p": 1.0,
    "B0": 4_000,
    "fish_init": 4_000,
    "sigma": 0.02,
    "initial_stock_log_sigma": 0.05,
    "unregulated_f_multiplier": 2.0,
}

REGULATOR_ECOLOGY = {"sustainability_weight": 2, "sustainability_threshold": 0.20}

## 6. Configure the regulator

The regulator is itself represented as an agent, but it has no learned policy. The evolution strategy searches the parameters of the mechanisms of this agent, here the single parameter of a `Quota` mechanism, and it does not use the `policy_id` of the agent. With several regulator agents it would search the union of their mechanisms, whose ids must then be distinct.

The argument

```python
acts_on=("fisherman", "harvest")
```

means that the institutional mechanism targets the fishers' harvest mechanism. The argument

```python
obs_map={"resource_level": "fish"}
```

tells the quota which entry of the society's state is the resource level. The quota runs inside the society's environment, so it reads the stock there and divides it by the carrying capacity $K$ of that environment. The regulator environment of the ES level is open loop: it never observes the stock, and it scores a candidate only from the metrics of the inner run.

The `default` of the mechanism, 0.56224, is the normalized quota that is played from the first reset until the first candidate is fetched from the evolution strategy. The search itself starts from the middle of the interval, 0.5, because the configuration sets no `initial_mean`. The evolution strategy evaluates `num_envs_per_env_runner // num_seeds` candidates per generation, which is 4 here (see the society configuration below).

The outer optimizer below is ES, but this is an optimizer choice rather than a property of the fishery benchmark. Its `episodes` is the number of generations.

The `reporting(queries=ES_QUERIES)` call lists only queries. The schema that the queries read is fixed by the optimizer class: an ES optimizer always logs an `ESSchema`, whose paths the queries in `ES_QUERIES` follow. The regulator environment has no schema of its own to declare. `ES_QUERIES` names the mechanism `"quota"` in its paths, so a mechanism with another id needs queries of its own.

In [ ]:
regulator_cfg = (
    ESConfig()
    .training(
        sigma=0.15,
        mean_lr=0.10,
        sigma_decay=1.0,
        sigma_lr=0.00,
        min_sigma=0.15,
        max_sigma=0.15,
        episodes=ES_GENERATIONS,
    )
    .agents(
        AgentConfig(
            id="fisheries_regulator",
            policy_id="quota_policy",
            mechanisms=(
                Quota(
                    id="quota",
                    action_space=spaces.Box(
                        low=0.0, high=1.0, shape=(1,), dtype=np.float32
                    ),
                    acts_on=("fisherman", "harvest"),
                    obs_map={"resource_level": "fish"},
                    default=np.asarray(0.56224),
                ),
            ),
        )
    )
    .environment(
        horizon=1,
        env=FisheryRegulatorEnv,
        env_config={"ecology_cfg": REGULATOR_ECOLOGY},
    )
    .debugging(seed=SEED)
    .reporting(queries=ES_QUERIES)
)

## 7. Configure the society

The society contains `NUM_FISHERS` copies of the same logical fisher agent. The `count` argument expands the entry into agents named `fisherman:0`, `fisherman:1` and so on, which all carry the policy id `"fisher_policy"`. Because they share that id, they share one policy module. The module that acts is named `<policy id>_m<candidate>_s<seed>`, so there is one module per candidate of the regulator and per training seed, and the fishers of one environment all use the module of their candidate.

The `shared_policy=True` flag documents this intent, but it is stored only: no code of the framework reads it today. The sharing comes from the common `policy_id`.

Each fisher owns two action mechanisms:

1. `Fishing(id="harvest")`
2. `Restore(id="restore")`

Each mechanism contributes one scalar action component. The policy action is a dictionary keyed by mechanism id, here with the entries `"harvest"` and `"restore"`, and each entry is decoded by its own mechanism.

In [ ]:
fisher = FishermanConfig(
    id="fisherman",
    policy_id="fisher_policy",
    shared_policy=True,
    count=NUM_FISHERS,
    mechanisms=(
        FishingConfig(
            id="harvest",
            action_space=spaces.Box(
                low=-np.inf, high=np.inf, shape=(1,), dtype=np.float32
            ),
        ),
        RestoreConfig(
            id="restore",
            action_space=spaces.Box(
                low=-np.inf, high=np.inf, shape=(1,), dtype=np.float32
            ),
        ),
    ),
    observation_space=spaces.Box(
        low=-np.inf, high=np.inf, shape=(5,), dtype=np.float32
    ),
)

The APPO configuration below is the lower-level learning problem.

The callback pair is required, and not only logging. `tag_episode_with_env_idx` rewrites the id of each new episode into `env=<index>|m=<mechanism>|ps=<policy seed>|ss=<environment seed>|raw=<id>`, and the policy mapping function of the society reads that id to route every agent to the module of its candidate and training seed. Without the tag the policies are not routed. `log_and_report_episode_metrics` then forwards the metrics of each finished episode into the reporting stack.

The environment horizon and the rollout fragment length are set to the same value, `HORIZON`, as a choice and not as a requirement: RLlib accepts a fragment length that differs from the episode length. With `batch_mode="truncate_episodes"`, each call to sample takes `rollout_fragment_length` steps from every sub-environment and continues the running episode in the next call. A fragment of exactly one episode therefore ends where an episode ends. A shorter fragment splits an episode over several calls, and a longer one ends in the middle of the next episode.

Reporting has two kinds of declarations. The queries of each optimizer go through `reporting(queries=...)`, and the schema of an optimizer is not a parameter: the RLlib-based society always logs a `RaySchema`, whose paths `INNER_QUERIES` follows. The one schema you pass is the environment's own, `schema=FisheryMetricSchema` in `environment(...)`, because the fishery defines what its environment logs at every step. The `queries=()` next to it means that the environment draws no plot of its own in this run.

In [ ]:
society_cfg = (
    APPOptimizerConfig()
    .resources(num_cpus_for_main_process=1)
    .framework(framework="torch")
    .api_stack(
        enable_rl_module_and_learner=True, enable_env_runner_and_connector_v2=True
    )
    .environment(
        env=FisheryRegulatedEnv,
        env_config={"ecology_cfg": ECOLOGY, "seed": 0},
        horizon=HORIZON,
        disable_env_checking=False,
        schema=FisheryMetricSchema,
        queries=(),
    )
    .env_runners(
        num_env_runners=0,
        num_cpus_per_env_runner=1,
        num_gpus_per_env_runner=0,
        num_envs_per_env_runner=4,
        rollout_fragment_length=HORIZON,
        batch_mode="truncate_episodes",
        max_requests_in_flight_per_env_runner=1,
    )
    .learners(num_learners=0, num_gpus_per_learner=0)
    .callbacks(
        on_episode_created=tag_episode_with_env_idx,
        on_episode_end=log_and_report_episode_metrics,
    )
    .training(
        episodes=APPO_ITERATIONS,
        vtrace=True,
        circular_buffer_num_batches=4,
        circular_buffer_iterations_per_batch=1,
        broadcast_interval=1,
        timeout_s_sampler_manager=10,
        timeout_s_aggregator_manager=10,
        gamma=0.99,
        lr=0.001,
        train_batch_size_per_learner=100,
        minibatch_size=100,
        num_epochs=1,
        entropy_coeff=0.001,
        grad_clip=40.0,
    )
    .evaluation(
        evaluation_interval=1,
        evaluation_duration=1,
        evaluation_duration_unit="episodes",
        evaluation_num_env_runners=0,
        evaluation_parallel_to_training=False,
        evaluation_config={
            "explore": False,
            "rollout_fragment_length": HORIZON,
            "batch_mode": "complete_episodes",
            "max_requests_in_flight_per_env_runner": 1,
        },
        base_seed=SEED,
        num_seeds=EVALUATION_SEEDS,
    )
    .agents(fisher)
    .fault_tolerance(restart_failed_env_runners=False)
    .debugging(seed=SEED, num_seeds=1)
    .reporting(
        min_time_s_per_iteration=0,
        min_sample_timesteps_per_iteration=0,
        min_train_timesteps_per_iteration=0,
        queries=INNER_QUERIES,
    )
)

## 8. Compose the bilevel experiment

`BilevelConfig` is the top-level composition object.

The reporter is declared here, once. The bilevel configuration gives a copy of it to each optimizer and to each environment, labelled by its owner, so that the outputs of the two levels stay apart. The regulator and society remain independently configured optimizer objects.

The reporter below is Weights & Biases. The notebook runs with `WANDB_MODE=offline`, which needs no account and writes the run to a local directory; with another mode, W&B asks for a login. `CSVConfig(project="bilevel")` is the login-free alternative that writes one CSV file per query, and it is what `examples/bilevel_fishery/debug.py --reporter csv` uses.

In [ ]:
bilevel_cfg = (
    BilevelConfig()
    .world(world_name="fishery_world")
    .reporter(
        config=WandbConfig(
            project="bilevel",
            x_disable_stats=True,
            x_disable_meta=True,
            quiet=True,
            max_end_of_run_summary_metrics=0,
            max_end_of_run_history_metrics=0,
        )
    )
    .ray(
        device="cpu",
        num_cpus=4,
        omp_threads=1,
        logging_level="ERROR",
        runtime_env={
            "excludes": [
                "wandb/",
                ".git/",
                ".venv/",
                "__pycache__/",
                "ray_results/",
                "runs/",
            ]
        },
    )
    .regulator(regulator_cfg)
    .society(society_cfg)
)

## 9. Build and train

The API has two explicit steps: `build_optimizer()` creates the Ray runtime, the world and both optimizers, and `train()` runs the bilevel loop and returns the summary of the outer optimizer.

```python
optimizer = bilevel_cfg.build_optimizer()
summary = optimizer.train()
```

Call `ray.shutdown()` before building the optimizer when you re-run notebook cells, and again when the experiment ends, even after a failure. This avoids carrying a stale Ray runtime between notebook executions. The cell below does both. The library does not configure logging, so the cell first turns on the `INFO` level of the root logger. The evolution strategy then logs several lines per generation; the one that starts with `[ES] gen=` gives the best and the mean fitness of the generation. A non-finite fitness stops the run with a `RuntimeError`, so the log is there to show whether the fitness moves, not whether it is valid. After the run, the cell checks that no Ray runtime is left behind and that the best fitness is finite.

In [ ]:
# The library does not configure logging: switch it on to see each ES generation.
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)

ray.shutdown()

try:
    optimizer = bilevel_cfg.build_optimizer()
    summary = optimizer.train()
finally:
    ray.shutdown()

assert not ray.is_initialized()
assert np.isfinite(summary["best_fitness"])

print(f"generations run : {summary['episodes']}")
print(f"converged       : {summary['converged']}")
print(f"best fitness    : {summary['best_fitness']:.4f}")
print(f"best quota      : {summary['best_mechanism']} (normalized, in [0, 1])")

### Reading the outcome at this scale

`summary` is the dictionary of the evolution strategy: the number of generations run, whether the convergence rule stopped the search, the best fitness and the best mechanism, which is the normalized quota in $[0, 1]$. After 12 generations the search has barely started, so the numbers only show that the loop runs and yields a finite fitness; they are not an estimate of the optimal quota.

The inner learner curves are empty at this scale. RLlib's APPO learner reduces its statistics only once every 20 gradient updates (the library notes this in `core/adaptors/ray/utils.py`). On the other iterations the value loss, the total loss, the policy loss and the entropy are NaN, and the library records them as gaps, so that every series keeps one point per iteration. With the learner settings of this notebook, which give about one gradient update per iteration, a run of 8 iterations never reaches an update at which APPO reports, and the four learner plots of `INNER_QUERIES` show nothing. In a measured run of 23 iterations of the same configuration, the policy loss was NaN on iterations 0 to 19, had a value on iteration 20 and was NaN again afterwards. The other inner plots, such as the returns, the biomass and the harvest, are not affected. Runs of 20 inner iterations or more draw the learner curves, with one point every 20 updates, and the full-scale run of Section 10 does.

The notebook does not fix `PYTHONHASHSEED`, so a rerun is not bit-identical even though `SEED` is set. The order of the agents in the inference batches of RLlib depends on Python's string hash seed, and that order decides which agent receives which draw of the seeded exploration noise. A running kernel cannot change its own hash seed. The command-line entry points do it for you: `core/config/hash_seed.py` restarts the process with `PYTHONHASHSEED=0` when the variable is unset. To make this notebook reproducible, start Jupyter with the variable set, for example `PYTHONHASHSEED=0 uv run jupyter lab`, or run the experiment from the command line with `uv run python -m examples.bilevel_fishery.debug`.

## 10. Full-scale settings of the original experiment

The notebook above runs at a small scale. The original experiment, which is also the default of `examples/bilevel_fishery/debug.py`, used the following values. To launch the real experiment, set the five constants of Section 5 to them and run the notebook again.

```python
ES_GENERATIONS = 1_000   # regulator .training(episodes=...): outer ES generations
APPO_ITERATIONS = 50     # society .training(episodes=...): inner APPO iterations per generation
HORIZON = 100            # episode length of the fishery, in steps
NUM_FISHERS = 10         # number of fishers
EVALUATION_SEEDS = 3     # society .evaluation(num_seeds=...)
```

At full scale the inner learner runs $1000 \times 50 = 50\,000$ APPO iterations in total, one block of 50 for every ES generation.

The important idea is to **shrink the experiment, not change its semantics**: the small scale keeps the same model structure and only reduces the computational size, so that you can verify construction, rollout, logging, and complete bilevel updates before starting a long run.

## 11. How the fishery benchmark maps to MetaMARL

The benchmark is easier to extend when each responsibility stays in the right layer.

| Research concept | MetaMARL object | Fishery example |
|---|---|---|
| Shared dynamical system | Environment | `FisheryRegulatedEnv` |
| Individual decision maker | Agent config | `FishermanConfig` |
| Individual action semantics | Agent mechanism | `FishingConfig`, `RestoreConfig` |
| Institution / intervention | Regulator mechanism | `Quota` |
| Follower learning | Society optimizer | `APPOptimizerConfig` |
| Leader learning | Regulator optimizer | `ESConfig` |
| Coupling of both levels | Experiment config | `BilevelConfig` |
| Metrics | Typed schemas + queries | `FisheryMetricSchema`, `RaySchema`, `ESSchema` |

This separation is what makes the package reusable: replacing APPO, ES, the fishery dynamics, or the quota should not require rewriting the other layers.

## 12. Extending the tutorial

### Replace the society optimizer

Keep the environment and fisher mechanisms fixed, construct another optimizer config for the society, and pass it to:

```python
BilevelConfig().society(other_society_cfg)
```

### Replace the regulator optimizer

ES is the only outer optimizer in the package today, so this means writing one. Keep the quota mechanism and regulator environment fixed, and give the new optimizer a `train()` that returns the summary described in Section 3 (`episodes`, `converged`, `best_mechanism`, `best_fitness`) and a settable `batch_capacity`.

### Add another institutional mechanism

Add another mechanism to the regulator agent's `mechanisms=(...)` tuple. A restoration subsidy or a social-influence mechanism can therefore be composed with the quota without embedding that logic inside the fishery transition.

### Change the benchmark

Implement another environment and its agent/mechanism objects, then reuse the same regulator/society composition pattern.

The package boundary to preserve is that the environment dynamics, the agent policy, the mechanism intervention and the optimizer remain four separate things, each of which can be replaced without editing the other three.

## 13. Practical checks before a long run

Before launching a full experiment, verify that:

1. the carrying capacity $K$ is set in the society's `ecology_cfg`, and the regulator's `ecology_cfg` holds only objective settings, since the regulator environment reads no carrying capacity;
2. `rollout_fragment_length` is a value you chose on purpose with respect to the society horizon (equal to it, as here, or different, which RLlib allows);
3. the observation-space shape matches the observation emitted by `Fisherman`;
4. every mechanism has a unique `id` within its owning agent;
5. `acts_on=("fisherman", "harvest")` still matches the target agent and mechanism IDs;
6. a short run produces finite regulator fitness and finite fishery metrics;
7. Ray is cleanly shut down when a notebook run ends.

These checks catch configuration mismatches without coupling the tutorial to a particular optimizer implementation.

## 14. What to read next in the benchmark code

For a deeper implementation walkthrough, read these files in the following order. The file `examples/bilevel_fishery/regulated_env.py` covers the fisher observations, the rewards, the harvesting and restoration mechanisms and the stock transition. The file `examples/bilevel_fishery/regulator_env.py` defines the regulator objective, and `examples/bilevel_fishery/metric_schema.py` the typed environment metrics. The file `examples/bilevel_fishery/queries.py` holds the reporter queries, and `examples/bilevel_fishery/debug.py` is the same experiment as a script with command-line options for the scale. The document `docs/ARCHITECTURE.md` explains how the two levels, the world and the reporting stack fit together, and the optimizer configurations under `core.optimizers` hold the algorithm-specific settings.

The tutorial teaches how the package is composed, and those modules remain the source of truth for the benchmark implementation.

## References

- Espeholt, L., Soyer, H., Munos, R., et al. (2018). IMPALA: Scalable distributed deep-RL with importance weighted actor-learner architectures. arXiv:1802.01561.
- Hilborn, R., & Walters, C. J. (1992). *Quantitative Fisheries Stock Assessment: Choice, Dynamics and Uncertainty*. Chapman and Hall, New York.
- Pella, J. J., & Tomlinson, P. K. (1969). A generalized stock production model. *Bulletin of the Inter-American Tropical Tuna Commission*, 13(3), 421-458.
- Salimans, T., Ho, J., Chen, X., Sidor, S., & Sutskever, I. (2017). Evolution Strategies as a scalable alternative to reinforcement learning. arXiv:1703.03864.
- Schaefer, M. B. (1954). Some aspects of the dynamics of populations important to the management of the commercial marine fisheries. *Bulletin of the Inter-American Tropical Tuna Commission*, 1(2), 27-56.